Import Library

In [20]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Menekan warning legacy dari Keras
import warnings
warnings.filterwarnings('ignore')

# Cek GPU
print("Daftar GPU yang aktif:", tf.config.list_physical_devices('GPU'))

Daftar GPU yang aktif: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


Dataset FairFace

In [22]:
print("Read file FairFace...")

# 1. Membaca label dari file CSV
train_df = pd.read_csv('Dataset/FairFace/train_labels.csv')
val_df = pd.read_csv('Dataset/FairFace/val_labels.csv')

# Filter
gender_map = {'Male': 0, 'Female': 1}
race_map = {
    'White': 0, 
    'Black': 1, 
    'Indian': 2, 
    'East Asian': 3, 
    'Southeast Asian': 4, 
    'Middle Eastern': 5, 
    'Latino_Hispanic': 6
}

age_map = {
    '0-2': 1, '3-9': 6, '10-19': 15, '20-29': 25, 
    '30-39': 35, '40-49': 45, '50-59': 55, '60-69': 65, 'more than 70': 75
}

for df in [train_df, val_df]:
    df['gender'] = df['gender'].map(gender_map)
    df['race'] = df['race'].map(race_map)
    df['age'] = df['age'].map(age_map)

print(f"Total Data Training : {len(train_df)} gambar")
print(f"Total Data Validasi : {len(val_df)} gambar")

Read file FairFace...
Total Data Training : 86744 gambar
Total Data Validasi : 10954 gambar


Image Data

In [23]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# Konfigurasi generator (hanya rescale piksel menjadi 0-1)
train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Fungsi khusus pemisah 3 output
def create_multi_output_generator(dataframe, datagen):
    generator = datagen.flow_from_dataframe(
        dataframe=dataframe, 
        directory='Dataset/FairFace/', 
        x_col='file', 
        y_col=['age', 'gender', 'race'],
        target_size=IMG_SIZE, 
        batch_size=BATCH_SIZE, 
        class_mode='raw', 
        shuffle=True
    )
    while True:
        x_batch, y_batch = next(generator)
        yield x_batch, {
            'age_output': y_batch[:, 0],
            'gender_output': y_batch[:, 1],
            'race_output': y_batch[:, 2]
        }

# Eksekusi generator
print("Menyiapkan aliran data...")
train_gen = create_multi_output_generator(train_df, train_datagen)
val_gen = create_multi_output_generator(val_df, val_test_datagen)

# steps proses training
langkah_train = len(train_df) // BATCH_SIZE
langkah_val = len(val_df) // BATCH_SIZE

print("Gambar berhasil disiapkan!")

Menyiapkan aliran data...
Gambar berhasil disiapkan!


Arsitektur - MobileNetV2

In [24]:
print("Arsitektur MobileNetV2.")

# Load model dasar tanpa layer klasifikasi atas (include_top=False)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # Kunci beban (weights) agar tidak berubah saat awal training

# Layer penyambung
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)

# 3 Cabang Output
output_age = Dense(1, activation='linear', name='age_output')(x)
output_gender = Dense(1, activation='sigmoid', name='gender_output')(x)
# Gunakan 7 layer softmax untuk 7 kelas ras FairFace
output_race = Dense(7, activation='softmax', name='race_output')(x) 

# Menyatukan input dan output menjadi satu Model utuh
model = Model(inputs=base_model.input, outputs=[output_age, output_gender, output_race])
model.summary()

Arsitektur MobileNetV2.
Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 112, 112, 32  864         ['input_2[0][0]']                
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 112, 112, 32  128         ['Conv1[0][0]']                  
                                )                                   

Grafik Evaluasi

In [ ]:
import matplotlib.pyplot as plt

# Mengambil data log dari history training
acc_gender = history.history['gender_output_accuracy']
val_acc_gender = history.history['val_gender_output_accuracy']

acc_race = history.history['race_output_accuracy']
val_acc_race = history.history['val_race_output_accuracy']

mae_age = history.history['age_output_mae']
val_mae_age = history.history['val_age_output_mae']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc_gender) + 1)

# Membuat Kanvas Gambar dengan 4 Subplot
fig, axs = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Performa Model Multi-Output (FairFace)', fontsize=16, fontweight='bold')

# 1. Grafik Total Loss
axs[0, 0].plot(epochs, loss, 'b-', label='Training Loss (Total)')
axs[0, 0].plot(epochs, val_loss, 'r-', label='Validation Loss (Total)')
axs[0, 0].set_title('Grafik Total Loss (Semakin turun semakin baik)')
axs[0, 0].set_xlabel('Epochs')
axs[0, 0].set_ylabel('Loss')
axs[0, 0].legend()
axs[0, 0].grid(True, linestyle='--', alpha=0.6)

# 2. Grafik Akurasi Gender
axs[0, 1].plot(epochs, acc_gender, 'b-', label='Training Acc (Gender)')
axs[0, 1].plot(epochs, val_acc_gender, 'r-', label='Validation Acc (Gender)')
axs[0, 1].set_title('Akurasi Tebakan Gender (Semakin naik semakin baik)')
axs[0, 1].set_xlabel('Epochs')
axs[0, 1].set_ylabel('Accuracy')
axs[0, 1].legend()
axs[0, 1].grid(True, linestyle='--', alpha=0.6)

# 3. Grafik Akurasi Ras/Etnis
axs[1, 0].plot(epochs, acc_race, 'b-', label='Training Acc (Race)')
axs[1, 0].plot(epochs, val_acc_race, 'r-', label='Validation Acc (Race)')
axs[1, 0].set_title('Akurasi Tebakan Ras (7 Kelas)')
axs[1, 0].set_xlabel('Epochs')
axs[1, 0].set_ylabel('Accuracy')
axs[1, 0].legend()
axs[1, 0].grid(True, linestyle='--', alpha=0.6)

# 4. Grafik MAE Umur (Mean Absolute Error)
axs[1, 1].plot(epochs, mae_age, 'b-', label='Training MAE (Age)')
axs[1, 1].plot(epochs, val_mae_age, 'r-', label='Validation MAE (Age)')
axs[1, 1].set_title('Tingkat Kesalahan Tebak Umur (Semakin turun semakin baik)')
axs[1, 1].set_xlabel('Epochs')
axs[1, 1].set_ylabel('Error (Tahun)')
axs[1, 1].legend()
axs[1, 1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

Training Model

In [ ]:
# Kompilasi model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={
        'age_output': 'mae', 
        'gender_output': 'binary_crossentropy', 
        'race_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'age_output': 'mae', 
        'gender_output': 'accuracy', 
        'race_output': 'accuracy'
    }
)

print("Memulai proses Training. RTX akan ngebut...")

# Memulai Training
history = model.fit(
    train_gen,
    steps_per_epoch=langkah_train,
    validation_data=val_gen,
    validation_steps=langkah_val,
    epochs=20 
)

print("Training Selesai!")

Memulai proses Training. RTX akan ngebut...
Found 86744 validated image filenames.
Epoch 1/20
1639/2710 [=================>............] - ETA: 12:29 - loss: 12.5690 - age_output_loss: 10.0661 - gender_output_loss: 0.5858 - race_output_loss: 1.9171 - age_output_mae: 10.0661 - gender_output_accuracy: 0.6921 - race_output_accuracy: 0.2707

In [ ]:
folder_simpan = "models"
lokasi_file = os.path.join(folder_simpan, "model_deteksi_wajah.h5")

# Save Models
model.save(lokasi_file)